
# AgroUTMACH PIX
## NB05 — Zonas, ocupación espacial, movimiento relativo y mapa de concentración

Este notebook usa **los tracks ya calculados en NB04**.  
No vuelve a entrenar ni a ejecutar YOLO.

### Objetivos
1. dividir el campo visual en cuatro zonas;
2. calcular cuántas aves visibles ocupan cada zona por fotograma;
3. estimar un **índice de movimiento relativo** a partir del desplazamiento de centroides;
4. construir un mapa de concentración acumulada;
5. generar un video para feria con:
   - zonas;
   - conteo por zona;
   - zona de mayor concentración;
   - movimiento relativo;
   - heatmap espacial.

> Las zonas representan regiones del **campo visual de la cámara**, no áreas físicas calibradas en metros.  
> El movimiento es relativo en píxeles/segundo y **no debe interpretarse como estrés, enfermedad ni velocidad física real**.

**Entorno recomendado: CPU.**


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

from pathlib import Path
from collections import defaultdict, deque
import os, json, math, shutil, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/AgroUTMACH_PIX")

TRACK_CSV = ROOT / "05_RESULTADOS" / "NB04_TRACKING" / "NB04_tracks_por_frame.csv"
VIDEO_DRIVE = ROOT / "04_VIDEOS_FERIA" / "PIO_tracking_estable_original_20s.mp4"

RESULTS_OUT = ROOT / "05_RESULTADOS" / "NB05_ESPACIAL"
VIDEOS_OUT = ROOT / "04_VIDEOS_FERIA"

RESULTS_OUT.mkdir(parents=True, exist_ok=True)

assert TRACK_CSV.exists(), f"Falta {TRACK_CSV}"
assert VIDEO_DRIVE.exists(), f"Falta {VIDEO_DRIVE}"

print("✓ Archivos de entrada encontrados")
print("Tracks:", TRACK_CSV)
print("Video:", VIDEO_DRIVE)


## 1. Copiar el video al almacenamiento local de Colab

In [ ]:

VIDEO_LOCAL = Path("/content/PIO_tracking_estable_original_20s.mp4")

if not VIDEO_LOCAL.exists() or VIDEO_LOCAL.stat().st_size != VIDEO_DRIVE.stat().st_size:
    shutil.copy2(VIDEO_DRIVE, VIDEO_LOCAL)
    print("✓ Video copiado a /content")
else:
    print("↪ Video local ya disponible.")

tracks = pd.read_csv(TRACK_CSV)

cap = cv2.VideoCapture(str(VIDEO_LOCAL))
assert cap.isOpened(), "No se pudo abrir el video."

fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = n_frames / fps
cap.release()

print(f"Resolución: {width}x{height}")
print(f"FPS: {fps:.2f}")
print(f"Frames: {n_frames}")
print(f"Duración: {duration:.2f} s")
print(f"Registros de tracking: {len(tracks):,}")
print(f"IDs temporales: {tracks['track_id'].nunique():,}")



## 2. Calcular centroides y asignar zonas

Usaremos una malla 2 × 2:

- **Z1:** superior izquierda
- **Z2:** superior derecha
- **Z3:** inferior izquierda
- **Z4:** inferior derecha

La asignación se realiza con el centro de cada *bounding box*.


In [ ]:

tracks["cx"] = (tracks["x1"] + tracks["x2"]) / 2
tracks["cy"] = (tracks["y1"] + tracks["y2"]) / 2

mid_x = width / 2
mid_y = height / 2

def assign_zone(x, y):
    if x < mid_x and y < mid_y:
        return "Z1"
    elif x >= mid_x and y < mid_y:
        return "Z2"
    elif x < mid_x and y >= mid_y:
        return "Z3"
    return "Z4"

tracks["zone"] = [assign_zone(x, y) for x, y in zip(tracks["cx"], tracks["cy"])]

zone_counts = (
    tracks.groupby(["frame", "zone"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Z1", "Z2", "Z3", "Z4"], fill_value=0)
)

all_frames = pd.Index(range(n_frames), name="frame")
zone_counts = zone_counts.reindex(all_frames, fill_value=0)

zone_counts["total_visible"] = zone_counts[["Z1", "Z2", "Z3", "Z4"]].sum(axis=1)
zone_counts["dominant_zone"] = zone_counts[["Z1", "Z2", "Z3", "Z4"]].idxmax(axis=1)

for z in ["Z1", "Z2", "Z3", "Z4"]:
    zone_counts[f"{z}_pct"] = np.where(
        zone_counts["total_visible"] > 0,
        zone_counts[z] / zone_counts["total_visible"] * 100,
        0.0
    )

zone_csv = RESULTS_OUT / "NB05_ocupacion_por_zona_frame.csv"
zone_counts.reset_index().to_csv(zone_csv, index=False)

print("✓ Ocupación por zona calculada")
display(zone_counts.head())


## 3. Resumen espacial del clip

In [ ]:

zone_totals = tracks["zone"].value_counts().reindex(["Z1", "Z2", "Z3", "Z4"], fill_value=0)
zone_share = zone_totals / zone_totals.sum() * 100

summary_zones = pd.DataFrame({
    "zone": zone_totals.index,
    "detections_accumulated": zone_totals.values,
    "share_percent": zone_share.values,
    "mean_birds_per_frame": [zone_counts[z].mean() for z in zone_totals.index],
    "max_birds_per_frame": [zone_counts[z].max() for z in zone_totals.index],
})

summary_zones.to_csv(RESULTS_OUT / "NB05_resumen_zonas.csv", index=False)
display(summary_zones)

dominant_global = summary_zones.loc[summary_zones["share_percent"].idxmax(), "zone"]
print("Zona con mayor ocupación acumulada:", dominant_global)



## 4. Índice de movimiento relativo

Para cada ID temporal se calcula el desplazamiento de su centro entre observaciones consecutivas.

La magnitud se expresa en **píxeles/segundo** y después se transforma a un índice 0–100 para la visualización de feria.

Este índice es únicamente un indicador visual/comparativo dentro de este clip.


In [ ]:

t = tracks.sort_values(["track_id", "frame"]).copy()

t["prev_cx"] = t.groupby("track_id")["cx"].shift(1)
t["prev_cy"] = t.groupby("track_id")["cy"].shift(1)
t["prev_frame"] = t.groupby("track_id")["frame"].shift(1)

t["frame_gap"] = t["frame"] - t["prev_frame"]
t["disp_px"] = np.sqrt(
    (t["cx"] - t["prev_cx"])**2 +
    (t["cy"] - t["prev_cy"])**2
)

valid = t["frame_gap"].between(1, 3) & t["disp_px"].notna()
t.loc[valid, "speed_px_s"] = t.loc[valid, "disp_px"] * fps / t.loc[valid, "frame_gap"]

movement_frame = (
    t.loc[valid]
    .groupby("frame")["speed_px_s"]
    .median()
    .reindex(range(n_frames))
    .interpolate(limit_direction="both")
    .fillna(0)
)

# Escala robusta 0–100 usando percentiles 5 y 95
p5 = float(movement_frame.quantile(0.05))
p95 = float(movement_frame.quantile(0.95))
den = max(p95 - p5, 1e-9)

movement_index = ((movement_frame - p5) / den * 100).clip(0, 100)

movement_df = pd.DataFrame({
    "frame": range(n_frames),
    "time_seconds": np.arange(n_frames) / fps,
    "median_speed_px_s": movement_frame.values,
    "movement_index_0_100": movement_index.values,
})

movement_df.to_csv(RESULTS_OUT / "NB05_movimiento_relativo_frame.csv", index=False)

print(f"Mediana global: {movement_frame.median():.1f} px/s")
print(f"P95: {movement_frame.quantile(0.95):.1f} px/s")
print("✓ Índice relativo generado.")


## 5. Construir mapa de concentración acumulada

In [ ]:

# Heatmap de menor resolución para eficiencia
HEAT_W, HEAT_H = 320, 180
heat = np.zeros((HEAT_H, HEAT_W), dtype=np.float32)

xs = np.clip((tracks["cx"].values / width * HEAT_W).astype(int), 0, HEAT_W - 1)
ys = np.clip((tracks["cy"].values / height * HEAT_H).astype(int), 0, HEAT_H - 1)

for x, y in zip(xs, ys):
    heat[y, x] += 1

heat_blur = cv2.GaussianBlur(heat, (0, 0), sigmaX=7, sigmaY=7)

if heat_blur.max() > 0:
    heat_norm = heat_blur / heat_blur.max()
else:
    heat_norm = heat_blur

plt.figure(figsize=(14, 8))
plt.imshow(heat_norm, cmap="inferno", origin="upper", extent=[0, width, height, 0])
plt.axvline(mid_x, linewidth=1.5)
plt.axhline(mid_y, linewidth=1.5)
plt.text(width*0.25, height*0.08, "Z1", ha="center", fontsize=14)
plt.text(width*0.75, height*0.08, "Z2", ha="center", fontsize=14)
plt.text(width*0.25, height*0.92, "Z3", ha="center", fontsize=14)
plt.text(width*0.75, height*0.92, "Z4", ha="center", fontsize=14)
plt.colorbar(label="Concentración relativa acumulada")
plt.title("AgroUTMACH PIX — Mapa de concentración espacial")
plt.xlabel("Posición horizontal (px)")
plt.ylabel("Posición vertical (px)")
plt.tight_layout()

heatmap_png = RESULTS_OUT / "NB05_heatmap_concentracion.png"
plt.savefig(heatmap_png, dpi=180, bbox_inches="tight")
plt.show()

print("✓ Heatmap guardado:", heatmap_png)


## 6. Gráfica de ocupación por zona

In [ ]:

time_axis = np.arange(n_frames) / fps

plt.figure(figsize=(14, 6))
for z in ["Z1", "Z2", "Z3", "Z4"]:
    plt.plot(time_axis, zone_counts[z].values, label=z)

plt.xlabel("Tiempo (s)")
plt.ylabel("Aves visibles")
plt.title("AgroUTMACH PIX - Ocupación por zona")
plt.legend()
plt.tight_layout()

zones_plot = RESULTS_OUT / "NB05_ocupacion_zonas_tiempo.png"
plt.savefig(zones_plot, dpi=180, bbox_inches="tight")
plt.show()

print("✓ Gráfica:", zones_plot)



## 7. Renderizar video final: zonas + heatmap dinámico

Se usará un mapa de concentración **acumulado con memoria temporal**, para que la audiencia vea dónde se concentran las aves durante el clip.

El heatmap no representa temperatura corporal ni estrés.  
Representa **densidad espacial de detecciones**.


In [ ]:
FINAL_TEMP = Path("/content/PIO_zonas_heatmap_temp.mp4")
FINAL_OUT = VIDEOS_OUT / "PIO_zonas_heatmap_20s.mp4"

cap = cv2.VideoCapture(str(VIDEO_LOCAL))
writer = cv2.VideoWriter(
    str(FINAL_TEMP),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

assert writer.isOpened(), "No se pudo crear el video temporal."

by_frame = {int(k): g for k, g in tracks.groupby("frame")}

dynamic_heat = np.zeros((HEAT_H, HEAT_W), dtype=np.float32)

# Decaimiento: conserva memoria aproximada de varios segundos
DECAY = 0.965

frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    current = by_frame.get(frame_idx, None)

    # Decaimiento temporal
    dynamic_heat *= DECAY

    if current is not None and len(current) > 0:
        xs = np.clip(
            (current["cx"].values / width * HEAT_W).astype(int),
            0, HEAT_W - 1
        )
        ys = np.clip(
            (current["cy"].values / height * HEAT_H).astype(int),
            0, HEAT_H - 1
        )
        for x, y in zip(xs, ys):
            dynamic_heat[y, x] += 1.0

    smooth = cv2.GaussianBlur(dynamic_heat, (0, 0), sigmaX=6, sigmaY=6)
    mx = smooth.max()

    if mx > 0:
        norm = np.clip(smooth / mx * 255, 0, 255).astype(np.uint8)
        colored = cv2.applyColorMap(norm, cv2.COLORMAP_INFERNO)
        colored = cv2.resize(
            colored, (width, height), interpolation=cv2.INTER_LINEAR
        )

        # Más transparencia donde hay poca señal
        alpha_map = cv2.resize(
            norm, (width, height), interpolation=cv2.INTER_LINEAR
        ).astype(np.float32) / 255.0
        alpha_map = np.clip(alpha_map * 0.50, 0, 0.50)[..., None]

        frame = (
            frame.astype(np.float32) * (1 - alpha_map)
            + colored.astype(np.float32) * alpha_map
        ).astype(np.uint8)

    # Líneas de zonas
    cv2.line(
        frame,
        (int(mid_x), 0),
        (int(mid_x), height),
        (255, 255, 255),
        2
    )
    cv2.line(
        frame,
        (0, int(mid_y)),
        (width, int(mid_y)),
        (255, 255, 255),
        2
    )

    zrow = zone_counts.loc[frame_idx]
    counts_now = {
        z: int(zrow[z])
        for z in ["Z1", "Z2", "Z3", "Z4"]
    }

    dominant = max(counts_now, key=counts_now.get)
    movement_now = float(
        movement_df.loc[frame_idx, "movement_index_0_100"]
    )
    total_now = sum(counts_now.values())

    # --------------------------------------------------------
    # Panel informativo: Z1, esquina superior izquierda
    # --------------------------------------------------------
    panel_x1, panel_y1 = 20, 20
    panel_x2, panel_y2 = 760, 185

    overlay = frame.copy()
    cv2.rectangle(
        overlay,
        (panel_x1, panel_y1),
        (panel_x2, panel_y2),
        (0, 0, 0),
        -1
    )
    frame = cv2.addWeighted(overlay, 0.62, frame, 0.38, 0)

    cv2.putText(
        frame,
        "AgroUTMACH PIX | Analisis espacial",
        (40, 58),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.82,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )
    cv2.putText(
        frame,
        f"Aves visibles: {total_now}",
        (40, 92),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.70,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )
    cv2.putText(
        frame,
        f"Mayor concentracion: {dominant}",
        (40, 124),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.70,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )
    cv2.putText(
        frame,
        f"Movimiento relativo: {movement_now:4.0f}/100",
        (40, 154),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.66,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )
    cv2.putText(
        frame,
        "Heatmap = concentracion espacial, no temperatura",
        (40, 176),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.43,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )

    # Etiquetas de zonas.
    # Z1 se desplaza a la derecha para no superponerse con el panel.
    labels_pos = {
        "Z1": (800, 75),
        "Z2": (int(mid_x) + 45, 75),
        "Z3": (45, int(mid_y) + 75),
        "Z4": (int(mid_x) + 45, int(mid_y) + 75),
    }

    for z, pos in labels_pos.items():
        cv2.putText(
            frame,
            f"{z}: {counts_now[z]}",
            pos,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.95,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

assert frame_idx > 0, "No se procesó ningún fotograma."

cmd = (
    f'ffmpeg -y -hide_banner -loglevel error '
    f'-i "{FINAL_TEMP}" -c:v libx264 -preset fast -crf 20 '
    f'-pix_fmt yuv420p -an "{FINAL_OUT}"'
)

rc = os.system(cmd)
assert rc == 0 and FINAL_OUT.exists(), "Falló la codificación final."

# Actualizar también el video que usa el dashboard en Drive.
DASHBOARD_ASSET = (
    ROOT / "06_DASHBOARD" / "assets" / "04_zonas_heatmap_20s.mp4"
)
DASHBOARD_ASSET.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(FINAL_OUT, DASHBOARD_ASSET)

print("✓ Video final:", FINAL_OUT)
print("✓ Dashboard actualizado:", DASHBOARD_ASSET)


## 8. Resumen final para el dashboard

In [ ]:

summary = {
    "video_duration_seconds": float(duration),
    "frame_width": int(width),
    "frame_height": int(height),
    "zones": {
        row["zone"]: {
            "share_percent": float(row["share_percent"]),
            "mean_birds_per_frame": float(row["mean_birds_per_frame"]),
            "max_birds_per_frame": int(row["max_birds_per_frame"]),
        }
        for _, row in summary_zones.iterrows()
    },
    "global_dominant_zone": str(dominant_global),
    "movement": {
        "median_speed_px_s": float(movement_frame.median()),
        "p95_speed_px_s": float(movement_frame.quantile(0.95)),
        "note": "relative image-space movement; not physical speed or welfare diagnosis"
    },
    "heatmap_note": "spatial detection concentration; not thermal imaging",
    "output_video": str(FINAL_OUT),
}

with open(RESULTS_OUT / "NB05_resumen_espacial.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))



## 9. Archivos producidos

```text
04_VIDEOS_FERIA/
└── PIO_zonas_heatmap_20s.mp4

05_RESULTADOS/NB05_ESPACIAL/
├── NB05_ocupacion_por_zona_frame.csv
├── NB05_resumen_zonas.csv
├── NB05_movimiento_relativo_frame.csv
├── NB05_heatmap_concentracion.png
├── NB05_ocupacion_zonas_tiempo.png
└── NB05_resumen_espacial.json
```

### Siguiente etapa
**NB06 — consolidación de videos y datos para el dashboard final AgroUTMACH PIX.**


In [ ]:

print("=" * 74)
print("NB05 FINALIZADO CORRECTAMENTE")
print("Zona global de mayor concentración:", dominant_global)
print("Video:", FINAL_OUT)
print("=" * 74)
